<div dir="rtl">

# کارگاه جلسه ۳۰ — جمع‌بندی و مسیرهای ادامه

این دفترچه سه مسیر ادامه را با داده واقعی نشان می‌دهد و تمرین عملی کلاس را با پاسخ دارد. نوشتن این کدها در امتحان پرسیده نمی‌شود.

## ۱. قواعد انجمنی روی سبدهای واقعی

سبدهای مشتریان **فرانسوی** فروشگاه اینترنتی جلسه ۲۹ را به جدول صفر و یک (سطر = فاکتور، ستون = کالا) تبدیل می‌کنیم و قواعد دوعضوی را با پشتیبان دست‌کم ۵ درصد می‌یابیم. کالای «POSTAGE» (هزینه ارسال) را حذف می‌کنیم چون تقریباً در همه سبدها هست. برای داده بزرگ‌تر از کتابخانه mlxtend (`apriori` و `association_rules`) استفاده کنید.

</div>

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv"
tx = pd.read_csv(url).dropna(subset=["CustomerID"])
tx = tx[~tx["InvoiceNo"].astype(str).str.startswith("C") & (tx["Quantity"] > 0) & (tx["UnitPrice"] > 0)].copy()
from itertools import combinations

fr = tx[(tx["Country"] == "France") & (tx["StockCode"] != "POST")]
basket = pd.crosstab(fr["InvoiceNo"], fr["Description"]).gt(0)
print(f"baskets={basket.shape[0]}  products={basket.shape[1]}")

min_support = 0.05
item_support = basket.mean()
frequent = item_support[item_support >= min_support].index          # Apriori: only frequent items go on
rows = []
for a, b in combinations(frequent, 2):
    s = (basket[a] & basket[b]).mean()
    if s >= min_support:
        for x, y in [(a, b), (b, a)]:
            conf = s / item_support[x]
            rows.append({"if": x, "then": y, "support": s, "confidence": conf, "lift": conf / item_support[y]})
rules = pd.DataFrame(rows).sort_values("lift", ascending=False)
print(f"frequent items={len(frequent)}  rules={len(rules)}")
print(rules.head(6).round(3).to_string(index=False))

baskets=384  products=1543
frequent items=75  rules=68
                           if                          then  support  confidence   lift
 PACK OF 6 SKULL PAPER PLATES    PACK OF 6 SKULL PAPER CUPS    0.052       0.909 13.964
   PACK OF 6 SKULL PAPER CUPS  PACK OF 6 SKULL PAPER PLATES    0.052       0.800 13.964
CHILDRENS CUTLERY DOLLY GIRL    CHILDRENS CUTLERY SPACEBOY     0.065       0.893 12.698
  CHILDRENS CUTLERY SPACEBOY  CHILDRENS CUTLERY DOLLY GIRL     0.065       0.926 12.698
   ALARM CLOCK BAKELIKE GREEN     ALARM CLOCK BAKELIKE RED     0.081       0.816  8.467
    ALARM CLOCK BAKELIKE RED     ALARM CLOCK BAKELIKE GREEN    0.081       0.838  8.467


<div dir="rtl">

## ۲. تشخیص ناهنجاری با Isolation Forest

روی جدول RFM مشتریان، `IsolationForest` با نسبت ناهنجاری ۱ درصد مشتریان غیرعادی را علامت می‌زند. مقایسه میانه آن‌ها با بقیه نشان می‌دهد چرا غیرعادی‌اند.

</div>

In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv"
tx = pd.read_csv(url).dropna(subset=["CustomerID"])
tx = tx[~tx["InvoiceNo"].astype(str).str.startswith("C") & (tx["Quantity"] > 0) & (tx["UnitPrice"] > 0)].copy()
import numpy as np
from sklearn.ensemble import IsolationForest

tx["InvoiceDate"] = pd.to_datetime(tx["InvoiceDate"], format="%m/%d/%Y %H:%M")
tx["amount"] = tx["Quantity"] * tx["UnitPrice"]
ref = tx["InvoiceDate"].max() + pd.Timedelta(days=1)
rfm = tx.groupby("CustomerID").agg(recency=("InvoiceDate", lambda d: (ref - d.max()).days),
                                   frequency=("InvoiceNo", "nunique"), monetary=("amount", "sum"))
iso = IsolationForest(contamination=0.01, random_state=42).fit(np.log1p(rfm))
rfm["anomaly"] = iso.predict(np.log1p(rfm)) == -1
print(rfm.groupby("anomaly").median().round(1).assign(n=rfm["anomaly"].value_counts()))

         recency  frequency  monetary     n
anomaly                                    
False       51.0        2.0     663.9  4294
True         2.0       39.0   37142.2    44


<div dir="rtl">

## ۳. خوشه‌بندی متن با TF-IDF

توضیحات متنی کالاها را با `TfidfVectorizer` به بردار تبدیل و با k-Means خوشه‌بندی می‌کنیم. پرتکرارترین واژه‌های مرکز هر خوشه، **موضوع** آن را نشان می‌دهد.

</div>

In [3]:
import pandas as pd

url = "https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv"
tx = pd.read_csv(url).dropna(subset=["CustomerID"])
tx = tx[~tx["InvoiceNo"].astype(str).str.startswith("C") & (tx["Quantity"] > 0) & (tx["UnitPrice"] > 0)].copy()
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

desc = tx["Description"].dropna().drop_duplicates()
vec = TfidfVectorizer(stop_words="english", min_df=5)
D = vec.fit_transform(desc)
km = KMeans(8, n_init=10, random_state=42).fit(D)
terms = np.array(vec.get_feature_names_out())
print(f"products={D.shape[0]}  words={D.shape[1]}")
for c in range(8):
    top = terms[np.argsort(km.cluster_centers_[c])[::-1][:6]]
    print(f"cluster {c} (n={np.sum(km.labels_ == c):4d}): {', '.join(top)}")

products=3877  words=637
cluster 0 (n= 218): heart, decoration, hanging, love, zinc, holder
cluster 1 (n=2705): pink, vintage, red, bag, christmas, box
cluster 2 (n= 111): polkadot, bowl, pink, blue, red, umbrella
cluster 3 (n=  82): card, birthday, greeting, wallet, travel, holder
cluster 4 (n= 312): set, lights, paper, tins, 12, vintage
cluster 5 (n= 191): blue, french, sign, metal, door, flower
cluster 6 (n=  95): wall, art, clock, mirrored, diner, flower
cluster 7 (n= 163): glass, bracelet, jar, light, small, holder


<div dir="rtl">

## تمرین عملی کلاس (۲۵ دقیقه)

1. در مثال کامل تایتانیک (جنگل تصادفی با حداقل ۳ نمونه در برگ)، با پیش‌بینی‌های خارج از بخش داده آموزش (`cross_val_predict`) بزرگ‌ترین آستانه‌ای را بیابید که بازخوانی کلاس بازمانده دست‌کم ۰.۷۵ شود. سپس مدل را روی کل آموزش برازش دهید و با این آستانه روی آزمون ماتریس درهم‌ریختگی را با آستانه ۰.۵ مقایسه کنید.
2. برای پنج سبد مثال حل‌شده، با حداقل پشتیبان ۰.۶ چند مجموعه پرتکرار و چند قاعده دوعضوی باقی می‌ماند؟ لیفت قاعده نوشابه ← شیر را با دست حساب کنید.
3. در سبدهای فرانسوی، کالای هزینه ارسال (`StockCode` برابر POST) را **نگه دارید** و اطمینان و لیفت قواعد «کالای پرتکرار ← هزینه ارسال» را حساب کنید. چرا این قواعد ارزش تجاری ندارند؟

</div>

In [4]:
# Write your solution here

<div dir="rtl">

### پاسخ تمرین

</div>

In [5]:
from itertools import combinations
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, recall_score

# 1. Threshold for recall >= 0.75 chosen on out-of-fold training predictions
titanic = sns.load_dataset("titanic")
num, cat = ["age", "fare", "sibsp", "parch", "pclass"], ["sex", "embarked"]
X_train, X_test, y_train, y_test = train_test_split(titanic[num + cat], titanic["survived"], test_size=0.2,
                                                    stratify=titanic["survived"], random_state=42)
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), cat)])
model = make_pipeline(preprocess, RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=42))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
p_oof = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
thresholds = np.arange(0.05, 0.95, 0.01)
ok = [t for t in thresholds if recall_score(y_train, p_oof >= t) >= 0.75]
t_best = max(ok)
print(f"chosen threshold={t_best:.2f}")
p_test = model.fit(X_train, y_train).predict_proba(X_test)[:, 1]
for t in [0.5, t_best]:
    print(f"threshold={t:.2f}  recall={recall_score(y_test, p_test >= t):.3f}")
    print(confusion_matrix(y_test, p_test >= t))

# 2. Five baskets, min support 0.6
baskets = [["bread", "milk"], ["bread", "diapers", "beer", "eggs"], ["milk", "diapers", "beer", "cola"],
           ["bread", "milk", "diapers", "beer"], ["bread", "milk", "diapers", "cola"]]
onehot = pd.DataFrame([{item: 1 for item in b} for b in baskets]).notna()
sup = onehot.mean()
items = sup[sup >= 0.6].index
pairs = [(a, b) for a, b in combinations(items, 2) if (onehot[a] & onehot[b]).mean() >= 0.6]
print(f"frequent items={list(items)}  frequent pairs={pairs}  rules={2 * len(pairs)}")

# 3. Keep POSTAGE in the French baskets
url = "https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv"
tx = pd.read_csv(url).dropna(subset=["CustomerID"])
tx = tx[~tx["InvoiceNo"].astype(str).str.startswith("C") & (tx["Quantity"] > 0) & (tx["UnitPrice"] > 0)]
basket = pd.crosstab(tx.loc[tx["Country"] == "France", "InvoiceNo"],
                     tx.loc[tx["Country"] == "France", "Description"]).gt(0)
sup = basket.mean()
print(f"support of POSTAGE={sup['POSTAGE']:.3f}")
rows = []
for item in sup[(sup >= 0.15) & (sup.index != "POSTAGE")].index:
    conf = (basket[item] & basket["POSTAGE"]).mean() / sup[item]
    rows.append({"if": item, "confidence": conf, "lift": conf / sup["POSTAGE"]})
print(pd.DataFrame(rows).round(3).to_string(index=False))

chosen threshold=0.39


threshold=0.50  recall=0.652
[[102   8]
 [ 24  45]]
threshold=0.39  recall=0.739
[[90 20]
 [18 51]]
frequent items=['bread', 'milk', 'diapers', 'beer']  frequent pairs=[('bread', 'milk'), ('bread', 'diapers'), ('milk', 'diapers'), ('diapers', 'beer')]  rules=8


support of POSTAGE=0.771
                                 if  confidence  lift
            LUNCH BAG RED RETROSPOT       0.800 1.037
     PLASTERS IN TIN CIRCUS PARADE        0.879 1.139
   PLASTERS IN TIN WOODLAND ANIMALS       0.806 1.045
                 RABBIT NIGHT LIGHT       0.890 1.155
      RED TOADSTOOL LED NIGHT LIGHT       0.886 1.148
ROUND SNACK BOXES SET OF4 WOODLAND        0.935 1.213


<div dir="rtl">

### پاسخ‌های کلیدی

- **آستانه برای بازخوانی دست‌کم ۰.۷۵:** روی پیش‌بینی‌های خارج از بخش، بزرگ‌ترین آستانه مناسب **۰.۳۹** است. روی آزمون، بازخوانی از ۰.۶۵۲ (۴۵ از ۶۹) به **۰.۷۳۹** (۵۱ از ۶۹) می‌رسد، به قیمت افزایش مثبت کاذب از ۸ به ۲۰. بازخوانی آزمون کمی کمتر از هدف ۰.۷۵ است؛ آزمون ۶۹ بازمانده دارد و نوسان طبیعی است، نه دلیلی برای تنظیم دوباره آستانه با آزمون.
- **پنج سبد با پشتیبان ۰.۶:** چهار قلم پرتکرار (نان، شیر، پوشک، نوشیدنی) و چهار جفت پرتکرار (نان و شیر، نان و پوشک، شیر و پوشک، پوشک و نوشیدنی)؛ هر جفت دو قاعده می‌دهد، پس **۸** قاعده. نوشابه (۰.۴) حذف می‌شود. قاعده نوشابه ← شیر با پشتیبان ۰.۴ در حالت قبل: اطمینان ۱ و لیفت یک تقسیم بر ۰.۸ یعنی **۱.۲۵**.
- **هزینه ارسال:** این قلم در ۷۷ درصد سبدهای فرانسوی هست، پس هر قاعده‌ای که به آن ختم شود اطمینان بالایی دارد (۰.۸ تا ۰.۹۴)، اما لیفت‌ها نزدیک یک‌اند (۱.۰۴ تا ۱.۲۱). این قواعد چیزی جز «بیشتر سفارش‌ها هزینه ارسال دارند» نمی‌گویند و هیچ پیشنهاد فروشی نمی‌سازند؛ نمونه روشن «اطمینان بالا کافی نیست».
- **کارگاه‌های بالا:** قواعد با بیشترین لیفت میان کالاهای **یک خانواده** است (لیوان و بشقاب کاغذی جمجمه، قاشق‌چنگال دخترانه و پسرانه، ساعت رومیزی سبز و قرمز) که برای پیشنهاد «مکمل» در فروشگاه مناسب است. ۴۴ مشتری ناهنجار عمده‌خریدهای بسیار فعال‌اند (میانه ۳۹ خرید و ۳۷ هزار پوند)، نه تقلب؛ ناهنجاری لزوماً «بد» نیست. در خوشه‌بندی متن، یک خوشه بزرگ (۲۷۰۵ کالا) با واژه‌های عمومی و چند خوشه موضوعی کوچک (قلب و تزئین، کارت تبریک، تابلو و ساعت دیواری) پدید آمد؛ برای متن، تعداد خوشه و کلمات توقف را باید با دقت تنظیم کرد.

</div>